# Indian Premier League (IPL) Exploratory Data Analysis (EDA)
**Course**: Exploratory Data Analysis  
**Instructor**: Ali Hassan Sherazi  
**Student Project**  

This notebook covers the exploratory data analysis of the IPL datasets: `matches.csv` (match-level details) and `deliveries.csv` (ball-by-ball level details). We will perform data loading, cleaning, preprocessing, and visualize key distributions and trends.

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set plotting aesthetic
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

## 2. Load and Clean Datasets

In [ ]:
# Load matches
matches_path = os.path.join("..", "data", "matches.csv")
df_matches = pd.read_csv(matches_path)

# Load sample of deliveries for structure inspection
deliveries_path = os.path.join("..", "data", "deliveries.csv")
df_deliveries_sample = pd.read_csv(deliveries_path, nrows=1000)

print("Matches Shape:", df_matches.shape)
print("Deliveries Sample Shape:", df_deliveries_sample.shape)

### Preprocessing and Standardization
- Standardize seasons to clean 4-digit strings.
- Standardize team names to handle name changes/mergers (e.g. Delhi Daredevils to Delhi Capitals, Kings XI Punjab to Punjab Kings).
- Fill missing cities based on venue names.
- Fill winner missing values.

In [ ]:
# Seasons mapping
season_map = {"2007/08": "2008", "2009/10": "2010", "2020/21": "2020"}
df_matches['season'] = df_matches['season'].replace(season_map)

# Teams mapping
team_map = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
    "Royal Challengers Bengaluru": "Royal Challengers Bangalore"
}
for col in ['team1', 'team2', 'toss_winner', 'winner']:
    df_matches[col] = df_matches[col].replace(team_map)

# Parse dates
df_matches['date'] = pd.to_datetime(df_matches['date'], errors='coerce')

# Handle missing cities
venue_city_map = {
    "M.Chinnaswamy Stadium": "Bangalore",
    "M Chinnaswamy Stadium": "Bangalore",
    "Punjab Cricket Association Stadium, Mohali": "Chandigarh",
    "Punjab Cricket Association IS Bindra Stadium, Mohali": "Chandigarh",
    "Feroz Shah Kotla": "Delhi",
    "Arun Jaitley Stadium": "Delhi",
    "Eden Gardens": "Kolkata",
    "Wankhede Stadium": "Mumbai",
    "Rajiv Gandhi International Stadium, Uppal": "Hyderabad",
    "MA Chidambaram Stadium, Chepauk": "Chennai",
    "MA Chidambaram Stadium": "Chennai",
    "Dubai International Cricket Stadium": "Dubai"
}
df_matches['city'] = df_matches['city'].fillna(df_matches['venue'].map(venue_city_map)).fillna('Other')

# Winner fillna
df_matches['winner'] = df_matches['winner'].fillna('No Result')
df_matches['result_margin'] = df_matches['result_margin'].fillna(0)
df_matches['result'] = df_matches['result'].fillna('no result')

print("Cleaned unique seasons:", sorted(df_matches['season'].dropna().unique()))

## 3. Exploratory Data Analysis & Visualizations
We will now look at specific questions and generate the required charts.

### Q1: What is the distribution of Toss Decisions?
We use a Pie Chart to look at the proportion of matches where teams chose to bat vs field after winning the toss.

In [ ]:
toss_data = df_matches['toss_decision'].value_counts()
plt.figure(figsize=(6, 6))
plt.pie(toss_data, labels=toss_data.index, autopct='%1.1f%%', startangle=90, colors=['#8a2be2', '#00f2fe'])
plt.title("Toss Decision Distribution")
plt.show()

### Q2: How are Target Runs distributed?
A Histogram with KDE overlay showing the frequency distribution of target runs set by teams batting first.

In [ ]:
sns.histplot(data=df_matches, x='target_runs', kde=True, color='#ffd700', bins=25)
plt.title("Distribution of Target Runs")
plt.xlabel("Target Score")
plt.ylabel("Count")
plt.show()

### Q3: How has the volume of matches changed over seasons?
A Line Chart tracking total matches played in each season.

In [ ]:
matches_per_season = df_matches['season'].value_counts().sort_index()
plt.plot(matches_per_season.index, matches_per_season.values, marker='o', color='#00f2fe', linewidth=2.5)
plt.title("Matches Played Over Seasons")
plt.xlabel("Season")
plt.ylabel("Number of Matches")
plt.xticks(rotation=45)
plt.show()

### Q4: Which teams have the most wins in IPL history?
A Bar Chart of victory counts for the top franchises.

In [ ]:
wins = df_matches[df_matches['winner'] != 'No Result']['winner'].value_counts().head(10)
sns.barplot(x=wins.values, y=wins.index, palette='viridis')
plt.title("Top 10 Teams by Wins")
plt.xlabel("Wins Count")
plt.ylabel("Team")
plt.show()

### Q5: Is there a relationship between Target Runs and Result Margin?
A Scatter Plot comparing Target Runs against Victory Margin, styled by win method.

In [ ]:
sns.scatterplot(data=df_matches[df_matches['result'].isin(['runs', 'wickets'])], 
                x='target_runs', y='result_margin', hue='result', palette='Set1', alpha=0.8)
plt.title("Target Runs vs. Result Margin")
plt.xlabel("Target Runs")
plt.ylabel("Victory Margin")
plt.show()

### Q6: How do Victory Margins distribute between Run wins and Wicket wins?
A Box Plot illustrating the median, range, and outliers of win margins.

In [ ]:
sns.boxplot(data=df_matches[df_matches['result'].isin(['runs', 'wickets'])], x='result', y='result_margin', palette='Set2')
plt.title("Victory Margin Spread by Win Type")
plt.xlabel("Win Type")
plt.ylabel("Winning Margin")
plt.show()

### Q7: What are the correlations between match numerical parameters?
A Heatmap visualizing the correlation matrix of target runs, result margins, and overs.

In [ ]:
cols = ['target_runs', 'result_margin', 'target_overs']
corr = df_matches[cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Correlation Matrix of Match Variables")
plt.show()

### Q8: What is the cumulative win trajectory for the top 4 teams?
An Area Chart displaying the cumulative win totals over seasons.

In [ ]:
top_4_teams = df_matches[df_matches['winner'] != 'No Result']['winner'].value_counts().head(4).index.tolist()
team_season_wins = df_matches[df_matches['winner'].isin(top_4_teams)].groupby(['season', 'winner']).size().unstack(fill_value=0)
cum_wins = team_season_wins.cumsum()

plt.stackplot(cum_wins.index, [cum_wins[team] for team in top_4_teams], labels=top_4_teams, alpha=0.8)
plt.title("Cumulative Wins Trajectory for Top 4 Teams")
plt.xlabel("Season")
plt.ylabel("Cumulative Wins")
plt.legend(loc='upper left')
plt.xticks(rotation=45)
plt.show()

### Q9: Which cities have hosted the most IPL matches?
A Count Plot of match frequencies by host city.

In [ ]:
city_counts = df_matches['city'].value_counts().head(10)
sns.barplot(x=city_counts.values, y=city_counts.index, palette='crest')
plt.title("Top 10 Host Cities by Match Count")
plt.xlabel("Number of Matches")
plt.ylabel("City")
plt.show()

### Q10: How have Target Scores evolved over the years?
A Violin Plot showing target run spread and probability density across IPL seasons.

In [ ]:
sorted_seasons = sorted(df_matches['season'].unique())
sns.violinplot(data=df_matches, x='season', y='target_runs', order=sorted_seasons, palette='plasma')
plt.title("Target Runs Spread Over Seasons")
plt.xlabel("Season")
plt.ylabel("Target Score")
plt.xticks(rotation=45)
plt.show()

## 4. Key Insights Summary
1. **Toss Impact**: Historically, teams in IPL have a high preference for fielding first, and winning the toss correlates strongly with match victories.
2. **Target runs**: The median target is around 170. Seasons have witnessed a gradual rise in average target scores, particularly in post-2018 years due to flat batting decks and batsman intent.
3. **Team dominance**: Mumbai Indians and Chennai Super Kings are the most dominant franchises, with Rajasthan Royals and Kolkata Knight Riders following closely.
4. **Margins**: Matches won by runs show a high variance in winning margins (up to 146 runs), indicating large-scale dominance in some matches, whereas wicket wins are bounded between 1 to 10.